# Notebook 01 — Dataset global completo con features estructurales

**Objetivo**: integrar al dataset base las variables estructurales del
proveedor (RUP, proveedores_obra) y del proceso (proponentes_por_proceso)
que la v1 del modelo no usó. Producir `consolidado_global.parquet`.

**Inputs**:
- `../../data/contratos_adiciones_obra.parquet` (48.331 × 125) — base con cols ricas
- `../../data/proveedores_rup.parquet` (28.548 × 22)
- `../../data/proveedores_obra.parquet` (13.117 × 25)
- `../../data/proponentes_por_proceso_obra.parquet` (276.770 × 9)
- `../../data/contratos_depurado_modelo.parquet` (48.331 × 53) — referencia targets + flags

**Outputs**:
- `data/consolidado_global.parquet` (48.331 × ~70 cols)
- `data/cols_leakage.txt`
- `data/schema_features.csv`

## 1. Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

RAIZ_PROYECTO = Path('../..').resolve()
RAIZ          = Path('..').resolve()
DATA_RAW      = RAIZ_PROYECTO / 'data'
DATA          = RAIZ / 'data'
DATA.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print('Working dir:', RAIZ)

Working dir: C:\Users\angel\OneDrive\Escritorio\trabajo dirigido\extraccion-datos\modelo_predictivo_global


## 2. Carga del dataset base (con todas las cols crudas)

In [2]:
base = pd.read_parquet(DATA_RAW / 'contratos_adiciones_obra.parquet')
dep  = pd.read_parquet(DATA_RAW / 'contratos_depurado_modelo.parquet')
print(f'base (adiciones_obra): {base.shape}')
print(f'depurado:              {dep.shape}')

# Verificar que el orden de filas coincide para poder copiar targets
assert len(base) == len(dep) == 48331, 'Shape mismatch'

base (adiciones_obra): (48331, 125)
depurado:              (48331, 53)


## 3. Definir features estructurales a conservar del base

Selección manual basada en `docs/00_plan.md`. Se excluyen IDs, PII,
texto libre y leakage.

In [3]:
# Features ya disponibles en depurado (mantener tal cual)
FEATURES_DEPURADO = [
    'departamento', 'ciudad', 'orden', 'sector', 'rama', 'entidad_centralizada',
    'modalidad_de_contratacion', 'condiciones_de_entrega', 'es_grupo', 'es_pyme',
    'habilita_pago_adelantado', 'obligaci_n_ambiental', 'origen_de_los_recursos',
    'destino_gasto', 'valor_del_contrato', 'valor_de_pago_adelantado', 'estado_bpin',
    'g_nero_representante_legal', 'presupuesto_general_de_la_nacion_pgn',
    'sistema_general_de_participaciones', 'sistema_general_de_regal_as',
    'recursos_propios_alcald_as_gobernaciones_y_resguardos_ind_genas_',
    'recursos_de_credito', 'recursos_propios', 'tipo_de_cuenta',
    'el_contrato_puede_ser_prorrogado', 'documentos_tipo',
    'duracion_planificada_dias',
    # RUP v1 ya usadas
    'rup_idx_endeudamiento', 'rup_idx_liquidez', 'rup_ingresos',
    'rup_utilidad_neta', 'rup_multas',
]

# Features estructurales NUEVAS del base (no estaban en depurado)
FEATURES_NUEVAS_BASE = [
    'log_valor_contrato',
    'nacionalidad_representante_legal',
    'obligaciones_postconsumo',
    'espostconflicto',
    'pilares_del_acuerdo',
    'puntos_del_acuerdo',
    'codigo_de_categoria_principal',
    'localizaci_n',
    'tipodocproveedor',
    'codigo_proveedor',   # NO es feature: clave para joins (drop al final)
    'proceso_de_compra',  # backup de clave para join procesos
]

TARGETS = ['tiempo', 'presupuesto', 'alcance']  # binarios fuente de los targets

cols_a_traer = FEATURES_DEPURADO + FEATURES_NUEVAS_BASE
print('Cols a traer del base:', len(cols_a_traer))

# Verificar que todas existen en base
faltantes = [c for c in cols_a_traer if c not in base.columns]
if faltantes: print('FALTANTES en base:', faltantes)
else: print('Todas las cols existen en base ✓')

Cols a traer del base: 44
Todas las cols existen en base ✓


In [4]:
df = base[cols_a_traer + TARGETS].copy()
print('df inicial:', df.shape)

df inicial: (48331, 47)


## 4. Enriquecer con `proveedores_rup` (variables financieras nuevas)

Variables a traer (sin las que ya están en depurado):
- `rup_tamano`, `rup_empleados`, `rup_sanciones`, `rup_inhabilidad`
- `rup_activo_total`, `rup_patrimonio`

Join por `nit` del proveedor — pero el base tiene `codigo_proveedor`,
que mapea contra `proveedores_obra.codigo` y luego `proveedores_obra.nit`
vs `proveedores_rup.nit`. Encadenamos.

In [5]:
pov = pd.read_parquet(DATA_RAW / 'proveedores_obra.parquet')
print('proveedores_obra:', pov.shape)
print('cols:', pov.columns.tolist())

proveedores_obra: (13117, 25)
cols: ['codigo', 'nombre', 'nit', 'es_entidad', 'es_grupo', 'esta_activa', 'fecha_creacion', 'codigo_categoria_principal', 'descripcion_categoria_principal', 'telefono', 'fax', 'correo', 'direccion', 'pais', 'departamento', 'municipio', 'sitio_web', 'tipo_empresa', 'nombre_representante_legal', 'tipo_doc_representante_legal', 'n_mero_doc_representante_legal', 'telefono_representante_legal', 'correo_representante_legal', 'espyme', 'ubicacion']


In [6]:
rup = pd.read_parquet(DATA_RAW / 'proveedores_rup.parquet')
print('proveedores_rup:', rup.shape)
# Cols de interés del RUP
RUP_NUEVAS = ['nit', 'rup_tamano', 'rup_empleados', 'rup_sanciones',
              'rup_inhabilidad', 'rup_activo_total', 'rup_patrimonio']
rup_slice = rup[[c for c in RUP_NUEVAS if c in rup.columns]].drop_duplicates(subset=['nit'])
print('rup_slice:', rup_slice.shape)

proveedores_rup: (28548, 22)
rup_slice: (28548, 7)


In [7]:
# Map: codigo_proveedor (en base) → codigo (en pov) → nit → join con rup
pov_keys = pov[['codigo', 'nit', 'esta_activa', 'fecha_creacion',
                'pais', 'departamento']].drop_duplicates(subset=['codigo'])
pov_keys = pov_keys.rename(columns={
    'codigo': 'codigo_proveedor',
    'departamento': 'departamento_proveedor',
    'pais': 'pais_proveedor',
})

# Merge en df
df = df.merge(pov_keys, on='codigo_proveedor', how='left')
print('df tras merge pov_keys:', df.shape)
print('Cobertura esta_activa:', df['esta_activa'].notna().mean().round(3))

df tras merge pov_keys: (48331, 52)
Cobertura esta_activa: 0.665


In [8]:
# Ahora merge con rup_slice por nit
df = df.merge(rup_slice, on='nit', how='left')
print('df tras merge RUP estructural:', df.shape)
for c in ['rup_tamano','rup_empleados','rup_sanciones','rup_inhabilidad','rup_activo_total','rup_patrimonio']:
    cov = df[c].notna().mean() * 100
    print(f'  {c}: {cov:.1f}% cobertura')

df tras merge RUP estructural: (48331, 58)
  rup_tamano: 49.0% cobertura
  rup_empleados: 49.0% cobertura
  rup_sanciones: 49.0% cobertura
  rup_inhabilidad: 49.0% cobertura
  rup_activo_total: 48.0% cobertura
  rup_patrimonio: 48.0% cobertura


## 5. Derivar `años_empresa` desde `fecha_creacion`

In [9]:
df['fecha_creacion'] = pd.to_datetime(df['fecha_creacion'], errors='coerce')
# Referencia: fecha actual del análisis (2026 según contexto del proyecto)
REF = pd.Timestamp('2026-01-01')
df['anios_empresa'] = ((REF - df['fecha_creacion']).dt.days / 365.25).round(2)
df.drop(columns=['fecha_creacion'], inplace=True)
print('anios_empresa stats:')
print(df['anios_empresa'].describe().round(2))

anios_empresa stats:
count    32126.00
mean         5.61
std          2.67
min         -0.08
25%          3.45
50%          5.86
75%          7.86
max         10.87
Name: anios_empresa, dtype: float64


## 6. Derivar `n_proponentes_por_proceso` (proxy competencia)

Conteo de oferentes únicos por `id_procedimiento`. Mapear contra
`proceso_de_compra` del contrato (es el mismo identificador).

In [10]:
# Necesitamos puente: contratos.proceso_de_compra = CO1.BDOS.*
# proponentes.id_procedimiento = CO1.REQ.*
# Puente: procesos_contratacion.id_del_portafolio (BDOS) <-> id_del_proceso (REQ)

prop = pd.read_parquet(DATA_RAW / 'proponentes_por_proceso_obra.parquet')
procs = pd.read_parquet(DATA_RAW / 'procesos_contratacion_obra.parquet')
print(f'proponentes: {prop.shape}')
print(f'procesos:    {procs.shape}')

# Tabla puente: id_del_portafolio (BDOS) -> id_del_proceso (REQ)
puente = procs[['id_del_portafolio', 'id_del_proceso']].drop_duplicates(subset=['id_del_portafolio'])
print(f'puente filas: {len(puente)}')

# Conteo proponentes por id_procedimiento (REQ)
n_prop = (
    prop.groupby('id_procedimiento')['nit_proveedor']
        .nunique()
        .rename('n_proponentes_por_proceso')
        .reset_index()
)

# Unir conteo al puente para tener n_proponentes por BDOS
puente = puente.merge(n_prop, left_on='id_del_proceso', right_on='id_procedimiento', how='left')
puente = puente[['id_del_portafolio', 'n_proponentes_por_proceso']].rename(columns={'id_del_portafolio': 'proceso_de_compra'})
print(f'puente con conteos: {len(puente)}')
print(f'cobertura conteos no-nulos: {puente["n_proponentes_por_proceso"].notna().mean():.3f}')


proponentes: (276770, 9)
procesos:    (138112, 57)
puente filas: 76322


puente con conteos: 76322
cobertura conteos no-nulos: 0.281


In [11]:
df = df.merge(puente, on='proceso_de_compra', how='left')
cov = df['n_proponentes_por_proceso'].notna().mean() * 100
print(f'Cobertura n_proponentes_por_proceso: {cov:.1f}%')
print(df['n_proponentes_por_proceso'].describe().round(1))


Cobertura n_proponentes_por_proceso: 48.2%
count    23281.0
mean         6.7
std         10.3
min          1.0
25%          1.0
50%          3.0
75%          8.0
max        118.0
Name: n_proponentes_por_proceso, dtype: float64


## 7. Construir targets y aislar leakage

In [12]:
df['tuvo_atraso']     = df['tiempo'].astype(int)
df['tuvo_sobrecosto'] = df['presupuesto'].astype(int)
df.drop(columns=['tiempo','presupuesto','alcance'], inplace=True)

# Drop claves no-features
df.drop(columns=['codigo_proveedor', 'proceso_de_compra', 'nit'],
        errors='ignore', inplace=True)

print('Balance targets:')
print(df[['tuvo_atraso','tuvo_sobrecosto']].mean().round(4))

Balance targets:
tuvo_atraso        0.6048
tuvo_sobrecosto    0.8341
dtype: float64


In [13]:
# Documentar leakage prohibido para etapas siguientes
COLS_LEAKAGE = [
    # 14 originales del sprint padre
    'tiempo', 'presupuesto', 'alcance',
    'n_modif_general', 'n_tipos_distintos', 'n_tipo_no_definido', 'n_suspension',
    'tiene_cesion', 'tiene_conclusion', 'tiene_suspension',
    'dias_adicionados', 'dias_hasta_primera_adicion', 'ventana_adiciones_dias',
    'ratio_extension_duracion',
    # adicionales detectadas en base
    'fecha_de_fin_del_contrato', 'fecha_de_inicio_del_contrato', 'fecha_de_firma',
    'fecha_primera_adicion', 'fecha_ultima_adicion', 'fecha_de_notificaci_n_de_prorrogaci_n',
    'fecha_fin_liquidacion', 'fecha_inicio_liquidacion',
    'duraci_n_del_contrato', 'liquidaci_n',
    'n_adicion_valor', 'n_extension', 'n_cesion', 'n_conclusion',
    'n_reactivacion', 'n_modificaciones_total',
    'tiene_adicion_valor', 'tiene_extension_dias', 'tiene_modificacion',
    'valor_amortizado', 'valor_facturado', 'valor_pagado',
    'valor_pendiente_de', 'valor_pendiente_de_ejecucion', 'valor_pendiente_de_pago',
    'saldo_cdp', 'saldo_vigencia',
    'sobrecosto_ratio', 'estado_contrato', 'ultima_actualizacion', 'reversion',
]
(DATA / 'cols_leakage.txt').write_text('\n'.join(COLS_LEAKAGE), encoding='utf-8')
print(f'cols_leakage.txt: {len(COLS_LEAKAGE)} cols documentadas')

cols_leakage.txt: 45 cols documentadas


## 8. Schema de features (tipo + cobertura + cardinalidad)

In [14]:
TARGET_COLS = ['tuvo_atraso', 'tuvo_sobrecosto']
feat_cols = [c for c in df.columns if c not in TARGET_COLS]

schema = pd.DataFrame({
    'columna': feat_cols,
    'dtype':   df[feat_cols].dtypes.astype(str).values,
    'n_unicos': [df[c].nunique(dropna=True) for c in feat_cols],
    'n_nan':   df[feat_cols].isna().sum().values,
    'pct_cobertura': ((1 - df[feat_cols].isna().mean()) * 100).round(2).values,
})
schema['tipo_logico'] = schema['dtype'].apply(
    lambda d: 'numerico' if d.startswith(('float','int')) else 'categorico'
)
schema = schema.sort_values(['tipo_logico', 'pct_cobertura', 'columna']).reset_index(drop=True)
schema.to_csv(DATA / 'schema_features.csv', index=False, encoding='utf-8')
print('Schema:', schema.shape)
schema

Schema:

 (53, 6)


,columna,dtype,n_unicos,n_nan,pct_cobertura,tipo_logico
0,rup_activo_total,str,6431,25108,48.05,categorico
1,rup_patrimonio,str,6402,25108,48.05,categorico
2,rup_empleados,str,242,24648,49.00,categorico
3,rup_inhabilidad,str,3,24648,49.00,categorico
4,rup_tamano,str,6,24648,49.00,categorico
5,departamento_proveedor,str,36,16205,66.47,categorico
6,esta_activa,str,2,16205,66.47,categorico
7,pais_proveedor,str,3,16205,66.47,categorico
8,ciudad,str,617,0,100.00,categorico
9,codigo_de_categoria_principal,str,1506,0,100.00,categorico


## 9. Reportes finales

In [15]:
print('=== Cobertura features estructurales nuevas ===')
estructurales_nuevas = [
    'rup_tamano', 'rup_empleados', 'rup_sanciones', 'rup_inhabilidad',
    'rup_activo_total', 'rup_patrimonio',
    'esta_activa', 'anios_empresa', 'pais_proveedor', 'departamento_proveedor',
    'n_proponentes_por_proceso',
    'log_valor_contrato', 'nacionalidad_representante_legal',
    'obligaciones_postconsumo', 'espostconflicto',
    'pilares_del_acuerdo', 'puntos_del_acuerdo',
    'codigo_de_categoria_principal', 'localizaci_n', 'tipodocproveedor',
]
for c in estructurales_nuevas:
    if c in df.columns:
        cov = df[c].notna().mean() * 100
        unicos = df[c].nunique(dropna=True)
        print(f'  {c:38s} cobertura {cov:5.1f}%  | unicos {unicos}')

=== Cobertura features estructurales nuevas ===
  rup_tamano                             cobertura  49.0%  | unicos 6
  rup_empleados                          cobertura  49.0%  | unicos 242
  rup_sanciones                          cobertura  49.0%  | unicos 6
  rup_inhabilidad                        cobertura  49.0%  | unicos 3
  rup_activo_total                       cobertura  48.0%  | unicos 6431
  rup_patrimonio                         cobertura  48.0%  | unicos 6402
  esta_activa                            cobertura  66.5%  | unicos 2
  anios_empresa                          cobertura  66.5%  | unicos 1050
  pais_proveedor                         cobertura  66.5%  | unicos 3
  departamento_proveedor                 cobertura  66.5%  | unicos 36
  n_proponentes_por_proceso              cobertura  48.2%  | unicos 91
  log_valor_contrato                     cobertura 100.0%  | unicos 39773
  nacionalidad_representante_legal       cobertura 100.0%  | unicos 9
  obligaciones_postconsum

In [16]:
# Exportar consolidado
df.to_parquet(DATA / 'consolidado_global.parquet', index=False)
print(f'Exportado: consolidado_global.parquet {df.shape}')
print(f'Features (sin targets): {len(feat_cols)}')

Exportado: consolidado_global.parquet (48331, 55)


Features (sin targets): 53


In [17]:
# Resumen ejecutivo
resumen = {
    'n_contratos':           len(df),
    'n_features':            len(feat_cols),
    'tasa_atraso':           round(df['tuvo_atraso'].mean(), 4),
    'tasa_sobrecosto':       round(df['tuvo_sobrecosto'].mean(), 4),
    'cols_leakage_documentadas': len(COLS_LEAKAGE),
    'features_estructurales_nuevas': len([c for c in estructurales_nuevas if c in df.columns]),
    'features_categoricas': int((schema['tipo_logico']=='categorico').sum()),
    'features_numericas':   int((schema['tipo_logico']=='numerico').sum()),
}
for k, v in resumen.items(): print(f'  {k:35s} {v}')

  n_contratos                         48331
  n_features                          53
  tasa_atraso                         0.6048
  tasa_sobrecosto                     0.8341
  cols_leakage_documentadas           45
  features_estructurales_nuevas       20
  features_categoricas                41
  features_numericas                  12
